## GWT-V5.ipynb

In [ ]:
import xmf6
import flopy

In [ ]:
phys = dict(
    specific_discharge = 0.2,  # Specific discharge ($m s^{-1}$)
    hydraulic_conductivity = 1.0,  # Hydraulic conductivity ($m s^{-1}$)
    source_concentration = contenido_44,  # Source concentration (unitless)
    porosity = 0.5,  # Porosity of mobile domain (unitless)
    initial_concentration = 0.0,  # Initial concentration (unitless)
)
xmf6.nice_print(phys, "Parámetros físicos")

In [ ]:
print("1. Imprimiendo: conc_inicial desde GWT-V5")
print(conc_inicial)
# Parámetros de la simulación (flopy.mf6.MFSimulation)
init_f = {
    'sim_name' : "flow",
    'exe_name' : r"C:\Users\luiggi\Documents\GitSites\mf6_tutorial\mf6\windows\mf6",
    'sim_ws' : "output_flow_trans_1D"
}

# Parámetros para el tiempo (flopy.mf6.ModflowTdis)
tdis_f = {
    'units': "days", #"seconds",
    'nper' : 1,
    'perioddata': [(dt, 1, 1.0)] #[(120.0, 240, 1.0)] (duración del periodo,num.de.pasos.de.tiempo,multiplicador)
}

# Parámetros para la solución numérica (flopy.mf6.ModflowIms)
# OJO: definiremos el modelo de solución más adelante
ims_f = {} 

# Parámetros para el modelo de flujo (flopy.mf6.ModflowGwf)
gwf = { 
    'modelname': init_f["sim_name"],
    'save_flows': True
}

# Parámetros para la discretización espacial (flopy.mf6.ModflowGwfdis)
# --- Diccionario dis ---
dis = {
    'length_units' : "meters",
    'nlay': 1,  
    'nrow': 1,  
    'ncol': ncolt, 
    'delr': dely,
    'delc': delx,
    'top' : delz,
    'botm': 0.0 
}

# Parámetros para las condiciones iniciales (flopy.mf6.ModflowGwfic)
ic = {
    'strt': 1.0 #'strt':conc_inicial 
}

# Parámetros para las condiciones de frontera (flopy.mf6.ModflowGwfchd)
chd = {
    'stress_period_data': [[(0, 0, dis['ncol'] - 1), 1.0]],     
}

# Parámetros para las propiedades de flujo (flopy.mf6.ModflowGwfnpf)
npf = {
    'save_specific_discharge': True,
    'save_saturation': True,
    'icelltype':  0,
    'k': phys["hydraulic_conductivity"] # Hydraulic conductivity ($cm s^{-1}$) 
}

# Parámetros para las propiedades de los pozos (flopy.mf6.ModflowGwfwel)
q = phys["specific_discharge"] * dis['delc'] * dis['delr'] * dis['top']
well = {
    'stress_period_data': [[(0, 0, 0), q, phys["source_concentration"],]],
    'pname': "WEL-1",
    'auxiliary' : ["CONCENTRATION"],
#    'save_flows': True
}

# Parámetros para almacenar y mostrar la salida de la simulación (flopy.mf6.ModflowGwfoc)
oc = {
    'budget_filerecord': f"{init_f['sim_name']}.bud",
    'head_filerecord': f"{init_f['sim_name']}.hds",
    'saverecord': [("HEAD", "ALL"), ("BUDGET", "ALL")],
#    'printrecord': [("HEAD", "ALL")]
}

# --- Inicialización de la simulación ---
o_sim_f = xmf6.common.init_sim(silent = True, init = init_f, tdis = tdis_f, ims = ims_f)
o_gwf, packages = xmf6.gwf.set_packages(o_sim_f, silent = True,
                                        gwf = gwf, 
                                        dis = dis, ic = ic, chd = chd, npf = npf, oc = oc, well = well)
# --- Escritura de archivos ---
o_sim_f.write_simulation(silent = True)

# --- Ejecución de la simulación ---
#o_sim_f.run_simulation(silent = False) #Activar mensajes
o_sim_f.run_simulation(silent = True) #Desactivar mensajes

## Paso 3. Simulación del transporte: GWT

Para resolver los cuatro casos definimos la función `get_mst_dict()` que genera la información necesaria para el paquete MST.

In [ ]:
def get_mst_dict(phys):
    mst = dict(
        porosity = phys["porosity"],
        sorption = None,
        bulk_density = None,
        distcoef = None, 
        first_order_decay = None,
        decay = None,
        decay_sorbed = None
    )
    
    if phys["retardation_factor"] > 1.0:
        mst["sorption"] = "linear"
        mst["bulk_density"] = 1.0
        mst["distcoef"] = (phys["retardation_factor"] - 1.0) * phys["porosity"] / mst["bulk_density"]

    if phys["decay_rate"] != 0.0:
        mst["first_order_decay"] = True
        mst["decay"] = phys["decay_rate"]
        if phys["retardation_factor"] > 1.0:
            mst["decay_sorbed"] = phys["decay_rate"]

    return mst

### Determinación de cada caso

In [ ]:
long_disp = [0.1, 1.0, 1.0, 1.0]
reta_fact = [1.0, 1.0, 2.0, 1.0]
deca_rate = [0.0, 0.0, 0.0, 0.01]
dir_names = ['p01a','p01b','p01c','p01d']

case = 1

dirname = dir_names[case]

# Agregamos más parámetros físicos al diccionario phys
phys["longitudinal_dispersivity"] = long_disp[case]
phys["retardation_factor"] = reta_fact[case]
phys["decay_rate"] = deca_rate[case]
phys["dispersion_coefficient"] = phys["longitudinal_dispersivity"] * phys["specific_discharge"] / phys["retardation_factor"]

xmf6.nice_print(phys, "Parámetros físicos")
print("Caso: {}".format(dirname))

In [ ]:
# Parámetros de la simulación (flopy.mf6.MFSimulation)
init_t = {
    'sim_name' : "transport",
    'exe_name' : r"C:\Users\luiggi\Documents\GitSites\mf6_tutorial\mf6\windows\mf6", # Tomamos el valor de la simulación de flujo
    'sim_ws' : init_f["sim_ws"]      # Tomamos el valor de la simulación de flujo
}

# Parámetros para el tiempo (flopy.mf6.ModflowTdis)
tdis_t = {
    'units': "days", #"seconds",
    'nper' : 1,
    'perioddata': [(1.0, 1, 1.0)] #[(120.0, 240, 1.0)]
}

# Parámetros para la solución numérica (flopy.mf6.ModflowIms)
ims_t = {
    'linear_acceleration': "bicgstab"
}

# Parámetros para el modelo de transporte (flopy.mf6.ModflowGwt)
gwt = { 
    'modelname': init_t["sim_name"], 
    'save_flows': True
}

# Parámetros para la discretización espacial (flopy.mf6.ModflowGwtdis)
# Es el mismo dis utilizado en la simulación de flujo, no es necesario redefinirlo
nlay, nrow, ncol = 1, 1, ncolt #120
conc_inicial = np.loadtxt("conc_i.m3d")
conc_inicial = conc_inicial.reshape((nlay, nrow, ncol))
#conc_inicial = np.ones((nlay, nrow, ncol))
#conc_inicial[0,0,ncolp]=magp

########################################################
"""
archivo = "conc_i.m3d" 

if os.path.exists(archivo):
    print("🔄🔴🔴 Leyendo conc_inicial de conc_i.m3d")
    
    conc_inicial = np.loadtxt(archivo) #lee los valores de conc_i.m3d

    # Convertir el vector leído a una matriz 3D (nlay, nrow, ncol)
    conc_inicial = conc_inicial.reshape((nlay, nrow, ncol))
    print(conc_inicial)
"""
##########################################################
# Parámetros para las condiciones iniciales (flopy.mf6.ModflowGwtic)
ic_t = {
    'strt':conc_inicial #    'strt': 0.0
}

# Parámetros para MST (flopy.mf6.ModflowGwtmst)
mst = get_mst_dict(phys)

# Parámetros para ADV (flopy.mf6.ModflowGwtadv)
adv = {
    "scheme" : "TVD"
}

# Parámetros para DSP (flopy.mf6.ModflowGwtdsp)
dsp = {
    "xt3d_off" : True,
    "alh" : phys["longitudinal_dispersivity"],
    "ath1" : phys["longitudinal_dispersivity"],
}

# Parámetros para FMI (flopy.mf6.ModflowGwtfmi)
# Se define el nombre de los archivos desde donde se lee la
# información del modelo de flujo calculada previamente
fmi = {
    "packagedata" : [("GWFHEAD", f"{init_f['sim_name']}.hds", None),
                     ("GWFBUDGET", f"{init_f['sim_name']}.bud", None),
                    ]
}

# Parámetros para SSM (flopy.mf6.ModflowGwtssm)
ssm = {
    "sources" : [["WEL-1", "AUX", "CONCENTRATION"]]
}

# Parámetros para OBS (flopy.mf6.ModflowGwtobs)
# Estos son sitios de observación espaciales fijos, para distintos tiempos.
obs = {
    "digits" : 10, 
    "print_input" : True, 
    "continuous" : {
        "transporte.obs.csv": [
            ("X005", "CONCENTRATION", (0, 0, 0)),
            ("X405", "CONCENTRATION", (0, 0, 1)), #40
            ("X1105", "CONCENTRATION", (0, 0, 2)), #110
        ],
    }
}

# Parámetros para almacenar y mostrar la salida de la simulación (flopy.mf6.ModflowGwtoc)
oc_t = {
    'budget_filerecord': f"{init_t['sim_name']}.bud",
    'concentration_filerecord': f"{init_t['sim_name']}.ucn",
    'saverecord' : [("CONCENTRATION", "ALL"), ("BUDGET", "LAST")],
    'printrecord' : [("CONCENTRATION", "LAST"), ("BUDGET", "LAST")],
}

# --- Inicialización de la simulación ---
o_sim_t = xmf6.common.init_sim(silent = True, init = init_t, 
                               tdis = tdis_t, ims = ims_t)
#o_sim_t = xmf6.common.init_sim(silent = True, init = init_t, 
#                               tdis = tdis_t, ims = ims_t)
o_gwt, packagest = xmf6.gwt.set_packages(o_sim_t, silent = True,
                                        gwt = gwt, 
                                        dis = dis, ic = ic_t, 
                                        mst = mst, adv = adv, dsp = dsp, fmi = fmi, ssm = ssm, 
                                        oc = oc_t)

o_obs = xmf6.common.set_obs(o_gwt, obs, silent = True)

# --- Escritura de archivos ---
o_sim_t.write_simulation(silent = True)

# --- Ejecución de la simulación ---
o_sim_t.run_simulation(silent = True)
#o_sim.run_simulation(silent = False)

print("🔴🔴conc_inicial antes de Análisis de resultados GWT-V5")
#print(conc_inicial)


## Paso 4. Análisis de resultados.

In [ ]:
from matplotlib.ticker import FormatStrFormatter
# --- Recuperamos los resultados de flujo de la simulación ---
head = xmf6.gwf.get_head(o_gwf)
qx, qy, qz, n_q = xmf6.gwf.get_specific_discharge(o_gwf, text="DATA-SPDIS")

# --- Recuperamos los resultados de la concentración de la simulación ---
cobj = o_gwt.output.concentration()

# --- Recuperamos las coordenadas del dominio
x, y, z = o_gwf.modelgrid.xyzcellcenters
row_length = o_gwf.modelgrid.extent[1]

#--- Leemos la solución analítica de la ec. de transporte ---
"""
sol_path = 'analytic/' + dirname
a1_0 = np.load(sol_path + '/a1_x_0.npy')
a1_1 = np.load(sol_path + '/a1_x_1.npy')
a1_2 = np.load(sol_path + '/a1_x_2.npy')
"""

# ==============================
# 🔥 CÁLCULO DEL MÁXIMO GLOBAL
# ==============================
# Solución numérica a distintos tiempos
ctimes = [1.0] # [1.0, 10.0, 20.0, 30.0, 40.0]
ctimes = [o_sim_t.transport.output.concentration().times[-1]]


cmax = -np.inf
for t in ctimes:
    gwt_conc = xmf6.gwt.get_concentration(o_sim_t, t)
    cmax = max(cmax, np.max(gwt_conc))

#print("Máximo de concentración:", cmax)

# ==============================
# 🔴 PARA INDICAR CADA CUANTOS np SE VISUALIZA
# ==============================

ntpt += 1

print(f"ntpt = {ntpt}")

# Graficar cada np pasos: quedó al principio del CICLO
# nplot = 5

# ==============================
# Decidir si se dibuja
# ==============================

if ntpt == 1 or ntpt % nplot == 0:
# ==============================
# 🔴 PARA INDICAR CADA CUANTOS np SE VISUALIZA
# ==============================



    # --- Definición de la figura ---
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize =(6,6), height_ratios=[0.1,1,3])
    
    # --- Gráfica 1. Malla ---
    pmv = flopy.plot.PlotMapView(o_gwf, ax=ax1)
    pmv.plot_grid(colors='dimgray', lw=0.5)
    ax1.set_yticks(ticks=[0, 0.1])
    ax1.set_title("Mesh")
    
    # --- Gráfica 2. Carga hidráulica---
    ax2.plot(
        x[0],
        head[0, 0],
        marker=".",
        ls="-",
        mec="blue",
        mfc="none",
        markersize=1,
        label='Head'
    )
    
    # 🔥 Escala automática en X
    ax2.set_xlim(0, row_length)
    
    # 🔥 Escala automática en Y (head)
    hmin = np.min(head)
    hmax = np.max(head)
    ax2.set_ylim(hmin - 0.05*(hmax-hmin), hmax + 0.05*(hmax-hmin))
    
    ax2.set_xticks(ticks=np.linspace(0, row_length, 13))
    ax2.set_ylabel("Head")
    ax2.grid()
    ax2.xaxis.set_major_formatter(FormatStrFormatter('%.1f'))
    #ax2.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))
    
    # --- Gráfica 3. Concentración ---
    # Solución analítica (opcional)
    """
    ax3.plot(x[0], a1_0, c='k', label='Analytic')
    ax3.plot(x[0], a1_1, c='k')
    ax3.plot(x[0], a1_2, c='k')
    """
    
    # Solución numérica a distintos tiempos 
    # ctimes = [ ]
    iskip = 1
    t_final = ctimes[-1]
        
    globals()['ntpt'] = ntpt
    
    for t in ctimes:
    
        gwt_conc = xmf6.gwt.get_concentration(o_sim_t, t)
    
        # Concentración 1D
        c_plot = gwt_conc.flatten()
    
        # Construir eje X compatible
        x_plot = np.linspace(0, row_length, len(c_plot))
    
        print("x_plot.shape =", x_plot.shape)
        print("c_plot.shape =", c_plot.shape)
    
        ax3.plot(
            x_plot[::iskip],
            c_plot[::iskip],
            "-o",
            linewidth=1.5,
            markersize=4,
            label=f"COMPONENTE {m+1} : GWT: Iter = {ntpt}"
        )
    
    if 'ntpt' in globals():
        ax3.set_title(f"WMA Iter = {ntpt} | GWT(t_final = {t_final})")
    else:
        ax3.set_title(f"GWT. Time = {t_final}")
    
    ax3.legend(fontsize=9)
        
    # ==============================
    # 🔥 ESCALAS DINÁMICAS
    # ==============================
    ax3.set_xlim(0, row_length)   #rango eje X
    #ax3.set_ylim(-0.05 * cmax, 1.1 * cmax) #rango eje Y ####NNNNNNNNNNNNN##############
    ax3.set_ylim(-2.51155E-06 , cmax) #rango eje Y (-0.05 * cmax, 1.1 * cmax)
    
    ax3.set_xticks(ticks=np.linspace(0, row_length, 13)) #13 marcas en eje X
    ax3.set_xlabel("Distance (m)") # etiqueta del eje X como "Distance(m)"
    ax3.set_ylabel("Concentration") # etiqueta eje Y como " Concentración"
    ax3.grid(True) # activa cuadrícula en gráfica
    ax3.xaxis.set_major_formatter(FormatStrFormatter('%.1f'))
    #ax3.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))
    ax3.set_title(f"Case:{dirname}") # define título usando nombre del caso (dirname)
    ax3.legend(fontsize=9)  #leyenda en tamaño de letra 9
    
    plt.tight_layout() # ajusta eutomáticamente espacios entre figuras
    plt.show() # muestra gráfica en pantalla
    
    ###########################################################
    # Imprimiendo los valores de concentración del último tiempo
    gwt_conc = xmf6.gwt.get_concentration(o_sim_t, ctimes[0])
    np.set_printoptions(precision=1, suppress=True)
    
    print("🔴🔴 Concentración (gwt_conc) en el último paso de tiempo de GWT-V5.ipynb:")
    #print(gwt_conc)
    print(np.array2string(
        gwt_conc.flatten(),
        formatter={'float_kind': lambda x: f"{x:.6e}"}
    ))
    print("Concentración máxima:", cmax)


## Termina transporte conservativo

In [ ]:
print("✔✔✔ Actualizando (u_tilde.dat) (1r de u_tilde.dat después de ejecutar GWT-V5.ipynb")

archivo = "u_tilde.dat"

# ==============================
# 1. Leer archivo completo
# ==============================
with open(archivo, "r") as f:
    lineas = f.readlines()

# ==============================
# 2. Crear nuevo primer renglón
# ==============================
nueva_linea = " ".join(f"{c:.4f}" for c in gwt_conc) + "\n"

# ==============================
# 3. Sustituir SOLO el primer renglón
# ==============================
if len(lineas) > 0:
    lineas[0] = nueva_linea
else:
    raise ValueError("⚠️ u_tilde.dat está vacío, no se puede sustituir el primer renglón")

# ==============================
# 4. Guardar archivo
# ==============================
with open(archivo, "w") as f:
    f.writelines(lineas)

print("✔✔✔ Primer renglón de u_tilde sustituido correctamente después de GWT-V5.ipynb")

# ==============================
# 🔍 Verificación
# ==============================
with open(archivo, "r") as f:
    print("\n 🔴🔴 u_tilde actualizado (1r de u_tilde después de ejecutar GWT-V5.ipynb:)\n")
    #print(f.read())